# Crop-Irrigation-AI — Model Training Walkthrough

End-to-end training pipeline for the three core ML models:

1. **Crop Classification** — Random Forest (Optuna-tuned) on optical+SAR+index features
2. **Moisture-Stress Detection** — CNN on 64×64 spectral patches
3. **Water-Deficit Regression** — XGBoost fusing ET₀, soil moisture, and crop coefficients

> **Note:** This notebook uses synthetic feature data for demonstration when real
> satellite-derived features are not yet available. Replace the synthetic-data
> cells with `FeatureExtractor` calls once you have processed Sentinel-1/2 scenes.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import settings
from models import CropClassifier, MoistureStressDetector, WaterDeficitModel, CROP_LABELS

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 5)


## 1. Crop Classification — Random Forest

### 1.1 Load / simulate training features

In production, replace this cell with:

```python
from preprocessing import FeatureExtractor
extractor = FeatureExtractor(optical_stack, sar_stack, index_dir)
X, y = extractor.extract_training_samples()
```


In [ ]:
FEATURE_NAMES = (
    ["B02","B03","B04","B08","B8A","B11","B12"] +
    ["NDVI","EVI","SAVI","LAI","NDWI","NDMI","LSWI","MSI","BSI"] +
    ["VV_dB","VH_dB","RVI","SoilMoisture"]
)

n_classes = len(CROP_LABELS)
n_samples_per_class = 400
n_features = len(FEATURE_NAMES)

y = np.repeat(np.arange(n_classes), n_samples_per_class)
X = np.random.normal(loc=y[:, None] * 0.4, scale=1.0, size=(len(y), n_features)).astype(np.float32)

print(f"Feature matrix: X={X.shape}, y={y.shape}")
print(f"Classes: {CROP_LABELS}")


### 1.2 Hyper-parameter tuning (Optuna)

In [ ]:
clf = CropClassifier(
    model_path=settings.saved_models_dir / "crop_classifier_demo.pkl",
    feature_names=FEATURE_NAMES,
)

best_params = clf.tune(X, y, n_trials=15, cv_folds=3)
best_params


### 1.3 Train final model with k-fold CV

In [ ]:
metrics = clf.train(X, y, params=best_params, cv_folds=settings.model.cv_folds)
print(f"CV F1-macro: {metrics['cv_f1_macro_mean']:.4f} ± {metrics['cv_f1_macro_std']:.4f}")
print(f"Train accuracy: {metrics['train_accuracy']:.4f}")
print()
print(metrics['classification_report'])


### 1.4 SHAP feature importance

In [ ]:
shap_df = clf.explain(X, n_samples=300, out_dir=settings.reports_dir)

fig, ax = plt.subplots(figsize=(8, 6))
top = shap_df.head(12)
ax.barh(top['feature'], top['mean_abs_shap'], color='#1D9E75')
ax.invert_yaxis()
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('Top features driving crop classification')
plt.tight_layout()
plt.show()


## 2. Water-Deficit Regression — XGBoost

In [ ]:
# Synthetic deficit features: [NDVI, NDWI, NDMI, MSI, EVI, LSWI, soil_moisture, Kc, et0, precip, temp, humidity]
n = 1000
X_wd = np.random.uniform(0, 1, size=(n, 12)).astype(np.float32)
y_wd = (45 * (1 - X_wd[:, 6]) + 10 * X_wd[:, 7] + np.random.normal(0, 4, n)).astype(np.float32)
y_wd = np.clip(y_wd, 0, None)

wd_model = WaterDeficitModel(
    model_type="xgboost",
    model_path=settings.saved_models_dir / "water_deficit_demo.pkl",
)
wd_metrics = wd_model.train(X_wd, y_wd, cv_folds=5)
print(f"MAE: {wd_metrics['cv_mae_mean']:.2f} ± {wd_metrics['cv_mae_std']:.2f} mm")
print(f"R²:  {wd_metrics['cv_r2_mean']:.4f}")


In [ ]:
preds = wd_model.predict(X_wd[:200])
fig, ax = plt.subplots()
ax.scatter(y_wd[:200], preds, alpha=0.5, color='#378ADD', s=20)
lims = [0, max(y_wd[:200].max(), preds.max())]
ax.plot(lims, lims, 'k--', alpha=0.5)
ax.set_xlabel('Actual deficit (mm)')
ax.set_ylabel('Predicted deficit (mm)')
ax.set_title('Water-Deficit Model: Predicted vs Actual')
plt.tight_layout()
plt.show()


## 3. Moisture-Stress CNN (PyTorch)

> **Heads up:** training the CNN end-to-end requires PyTorch and is more
> compute-intensive. The cell below runs a short demonstration with synthetic
> patches at a reduced epoch count — increase `epochs` for real use.


In [ ]:
in_channels = 6
patch_size = 64
n_patches = 200
n_stress_classes = 4

patches = np.random.uniform(0, 1, size=(n_patches, in_channels, patch_size, patch_size)).astype(np.float32)
stress_labels = np.random.randint(0, n_stress_classes, size=n_patches).astype(np.int64)

detector = MoistureStressDetector(
    model_path=settings.saved_models_dir / "moisture_stress_demo.pt",
    in_channels=in_channels,
)

history = detector.train(patches, stress_labels, epochs=5, batch_size=16)
pd.DataFrame(history)


In [ ]:
hist_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist_df['epoch'], hist_df['train_loss'], color='#D85A30')
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')

axes[1].plot(hist_df['epoch'], hist_df['val_f1'], color='#1D9E75')
axes[1].set_title('Validation F1-macro')
axes[1].set_xlabel('Epoch')

plt.tight_layout()
plt.show()


## 4. Summary

| Model | Metric | Demo Result |
|---|---|---|
| Crop Classification | CV F1-macro | see Section 1.3 output |
| Water Deficit | CV MAE (mm) | see Section 2 output |
| Moisture Stress | Validation F1 | see Section 3 output |

All trained models are saved under `models/saved/`. Run `python main.py run`
to apply them across a full satellite scene and generate irrigation advisories.
